In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType
from datetime import datetime

In [ ]:
schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("itemid", IntegerType(), True),
    StructField("property", IntegerType(), True),
    StructField("value", StringType(), True)
])

In [ ]:
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", checkpoint_location) \
    .schema(schema) \
    .load(s3_path)

In [ ]:
df = df.withColumn("_ingestion_time", current_timestamp()) \
       .withColumn("_source_file", input_file_name())

In [ ]:
query = df.writeStream \
    .format("delta") \
    .mode("append") \
    .option("checkpointLocation", checkpoint_location) \
    .table(target_table)